In [2]:
import numpy as np

GRAVITY = 9.806

def category_to_kgf_per_m2(category_MPa):
    return category_MPa * 1e6 / GRAVITY


def cycles_to_failure(delta_sigma, delta_sigma_C,
                      sn_model="eurocode",
                      m1=3, m2=5,
                      N_C=2e6, N_D=5e6, N_L=1e8):
    """
    Cycles to failure for stress range Δσ at a given detail category, 
    can be in either Eurocode or AASHTO (ie bilinear or single line)
    """
    delta_sigma = np.asarray(delta_sigma, dtype=float)
    delta_sigma_safe = np.maximum(delta_sigma, 1e-12) # avoid dividing by zero

    delta_sigma_D = delta_sigma_C * (N_C / N_D) ** (1.0 / m1)
    delta_sigma_L = delta_sigma_D * (N_D / N_L) ** (1.0 / m2)

    if sn_model == "eurocode":
        N_high = N_C * (delta_sigma_C / delta_sigma_safe) ** m1
        N_low  = N_D * (delta_sigma_D / delta_sigma_safe) ** m2
        N = np.where(delta_sigma >= delta_sigma_D, N_high, N_low) #if stress is above knee, use upper slope, if not, use lower slope
        N = np.where(delta_sigma < delta_sigma_L, np.inf, N)
    elif sn_model == "aashto":
        A = N_C * delta_sigma_C ** m1
        N = A / (delta_sigma_safe ** m1)
        N = np.where(delta_sigma < delta_sigma_D, np.inf, N)
    else:
        raise ValueError("sn_model must be 'eurocode' or 'aashto'.")

    return N
